In [1]:
import py4cytoscape as p4c
p4c.cytoscape_ping()
p4c.cytoscape_version_info()

You are connected to Cytoscape!


{'apiVersion': 'v1',
 'cytoscapeVersion': '3.10.3',
 'automationAPIVersion': '1.11.0',
 'py4cytoscapeVersion': '1.11.0'}

In [2]:
import re
import math

In [3]:
node_table = p4c.get_table_columns('node')

# Copy label to name
node_table['name'] = node_table['label']

# Write back to Cytoscape
p4c.load_table_data(node_table, data_key_column="SUID", table_key_column="SUID")

'Success: Data loaded in defaultnode table'

In [4]:
#event_re = re.compile(r"^event_\d+$")
number_re = re.compile(r"\d+")
neuron_re = re.compile(r"^n\d+_\d+$")
weight_re1 = re.compile(r"^w\d+_\d+_\d+$")
weight_re2 = re.compile(r"^w\d+_\d+$")

names = p4c.networks.get_all_nodes()
weights = [node for node in names if re.match(weight_re1, node) or re.match(weight_re2, node)]
p4c.hide_nodes(weights + ["litre_base", "LITRE", "m2", "METRE", "mmol", "MOLE"])

neurons = [node for node in names if re.match(neuron_re, node)]
neurons_layers = {}
for neuron in neurons:
    layer_num = int(re.findall(number_re, neuron)[0])
    if layer_num in neurons_layers.keys():
        neurons_layers[layer_num].append(neuron)
    else:
        neurons_layers[layer_num] = [neuron]
sorted_keys = sorted(neurons_layers)
def sorting_fun2(s):
    num = re.findall(number_re, s)[1]
    return int(num)
sorted_dict = {key: neurons_layers[key] for key in sorted_keys}
for val in sorted_dict.values():
    val.sort(key = sorting_fun2)

x0 = 0
y0 = 0
dx = 200
dy = 30
d_ea = 40

to_pos = []
xs = []
ys = []
#events_to_interm = []


for key, val in sorted_dict.items():
    for i, name in enumerate(val):
        to_pos.append(name)
        xs.append(x0 + key*dx)
        ys.append(y0 + i*dy)
        to_pos.append(f"<{name}>")
        xs.append(x0 + key*dx - d_ea)
        ys.append(y0 + i*dy)


input_output_dy = dy*3
outputs = ["X", "S", "PT", "PM", "V"]
output_x0 = x0 + dx*len(list(sorted_dict.keys()))
output_y0 = y0 + (dy*(len(sorted_dict[list(sorted_dict.keys())[-1]]) - 1) - input_output_dy*(len(outputs) - 1))/2
output_dx = 60

layer_name = f"c"
to_pos.append(layer_name)
xs.append(output_x0 + 3*output_dx)
ys.append(y0 - 3*dy)

for i, output in enumerate(outputs):
    to_pos.append(f"<{output}>")
    xs.append(output_x0 + 2*output_dx)
    ys.append(output_y0 + i*input_output_dy)
    to_pos.append(f"{output}_max")
    xs.append(output_x0 + 2*output_dx)
    ys.append(output_y0 + i*input_output_dy - dy*0.7)
    to_pos.append(f"{output}_min")
    xs.append(output_x0 + 2*output_dx)
    ys.append(output_y0 + i*input_output_dy + dy*0.7)
    to_pos.append(output)
    xs.append(output_x0 + 3*output_dx)
    ys.append(output_y0 + i*input_output_dy)

inputs = ["t", "X0", "S0", "V0", "Feed0"]
time_additions = ["t_scaled", "<t_scaled>", "t_max"]
x0_input = x0
y0_input = y0
for i, name in enumerate(inputs):
    if name == "t":
        for j, addit in enumerate(time_additions):
            to_pos.append(addit)
            xs.append(x0_input - j*d_ea)
            ys.append(y0_input)
    else:
        to_pos.append(f"{name}_scaled")
        xs.append(x0_input)
        ys.append(y0_input + i*input_output_dy)
        to_pos.append(f"assignment__{name}_scaled")
        xs.append(x0_input - d_ea)
        ys.append(y0_input + i*input_output_dy)
        to_pos.append(f"{name}_max")
        xs.append(x0_input - d_ea)
        ys.append(y0_input + i*input_output_dy - dy*0.7)
        to_pos.append(f"{name}_min")
        xs.append(x0_input - d_ea)
        ys.append(y0_input + i*input_output_dy + dy*0.7)
        to_pos.append(name)
        xs.append(x0_input - 2*d_ea)
        ys.append(y0_input + i*input_output_dy)


p4c.set_node_position_bypass(to_pos, new_x_locations=xs, new_y_locations=ys)

''